

# # AgriSense — Task 5: Multi-Temporal Change Detection & Crop Stress Alerts
#
# **Pipeline:** NDVI Differencing → Log-Ratio Change Maps → Change Vector Analysis (CVA)
#              → Siamese CNN (PyTorch) → Alert Shapefile Export (GeoPandas/Fiona)
#
# **Outputs:**
# - Output A: GeoTIFF binary change mask (EPSG:32642)
# - Output B: Shapefile of alert polygons (area_ha, severity, GPS centroid)
# - Output C: Matplotlib map overlay for stakeholder report
#
# **Metrics:** Producer's Accuracy · User's Accuracy · OA · Cohen's κ · FAR · MDR

In [1]:
import os
import time
import warnings
import csv
from pathlib import Path
from datetime import datetime

warnings.filterwarnings("ignore")

In [2]:
# ── Numerical / geospatial ────────────────────────────────────────────────────
import numpy as np
import rasterio
from rasterio.transform import from_bounds
from rasterio.features import shapes
from rasterio.crs import CRS
import geopandas as gpd
from shapely.geometry import shape, mapping

# ── Machine learning ──────────────────────────────────────────────────────────
from sklearn.metrics import cohen_kappa_score, confusion_matrix
from scipy.ndimage import label as scipy_label
from scipy.ndimage import binary_opening, binary_closing

# ── Deep learning ─────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib
matplotlib.use("Agg")                # headless-safe
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap, BoundaryNorm
import matplotlib.gridspec as gridspec

# ── Output directory ──────────────────────────────────────────────────────────
OUT = Path("task5_outputs")
OUT.mkdir(exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[T5] Device: {DEVICE} | Output dir: {OUT.resolve()}")


[T5] Device: cuda | Output dir: /content/task5_outputs


In [5]:
# 5.1 — Synthetic Bi-Temporal Data Generation
#
# In production replace with:
# ```python
with rasterio.open("/content/drive/MyDrive/AgriSense/S2_final_stack.tif") as src:
 stack_t1 = src.read().transpose(1,2,0).astype(np.float32)
 # (H,W,C)
#     TRANSFORM,
 CRS_VAL = src.transform, src.crs
# ```

# %%
def simulate_sentinel2_scene(H: int, W: int, C: int, seed: int) -> np.ndarray:
    """Generate a realistic synthetic Sentinel-2 BOA reflectance scene (0–1).

    Band order (C=10): B02(Blue), B03(Green), B04(Red),
                        B05(RE1),  B06(RE2),  B07(RE3),
                        B08(NIR),  B8A(NRE),  B11(SWIR1), B12(SWIR2)
    """
    rng = np.random.default_rng(seed)
    H_half, W_half = H // 2, W // 2

    # Baseline reflectance (spectral profile per land cover)
    LC_PROFILES = {
        "wheat":     [0.05, 0.07, 0.04, 0.10, 0.20, 0.30, 0.40, 0.42, 0.22, 0.14],
        "cotton":    [0.06, 0.09, 0.05, 0.12, 0.25, 0.35, 0.45, 0.46, 0.25, 0.16],
        "fallow":    [0.15, 0.17, 0.18, 0.20, 0.22, 0.23, 0.24, 0.24, 0.30, 0.28],
        "water":     [0.03, 0.05, 0.03, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01],
    }
    scene = np.zeros((H, W, C), dtype=np.float32)

    # Quadrant assignment
    quadrant_lc = [
        ("wheat",  slice(0,  H_half), slice(0,  W_half)),
        ("cotton", slice(0,  H_half), slice(W_half, W)),
        ("fallow", slice(H_half, H), slice(0,  W_half)),
        ("water",  slice(H_half, H), slice(W_half, W)),
    ]
    for lc, rs, cs in quadrant_lc:
        profile = np.array(LC_PROFILES[lc], dtype=np.float32)
        noise   = rng.normal(0, 0.01, size=(rs.stop-rs.start, cs.stop-cs.start, C)).astype(np.float32)
        scene[rs, cs] = profile + noise

    return np.clip(scene, 0.0, 1.0)


# ── Scene parameters ──────────────────────────────────────────────────────────
H, W, C = 256, 256, 10
EPSG_UTM = "EPSG:32642"
TRANSFORM = from_bounds(73.0, 30.0, 73.5, 30.5, W, H)   # Punjab, Pakistan

# ── T1 = healthy / T2 = after stress event ────────────────────────────────────
stack_t1 = simulate_sentinel2_scene(H, W, C, seed=42)
stack_t2 = simulate_sentinel2_scene(H, W, C, seed=99)

# Introduce synthetic stress: NDVI drop in wheat quadrant (top-left)
STRESS_RS = slice(20, 100)
STRESS_CS = slice(20, 100)
# Suppress NIR (band 6) and boost Red (band 2) → NDVI drop
stack_t2[STRESS_RS, STRESS_CS, 6] *= 0.45   # NIR ↓
stack_t2[STRESS_RS, STRESS_CS, 2] *= 1.60   # Red ↑  (clip below)
stack_t2 = np.clip(stack_t2, 0.0, 1.0)

print(f"[T5.1] stack_t1 shape: {stack_t1.shape}  |  stack_t2 shape: {stack_t2.shape}")
print(f"[T5.1] Value range T1 — min={stack_t1.min():.4f}  max={stack_t1.max():.4f}")
print(f"[T5.1] Value range T2 — min={stack_t2.min():.4f}  max={stack_t2.max():.4f}")

[T5.1] stack_t1 shape: (256, 256, 10)  |  stack_t2 shape: (256, 256, 10)
[T5.1] Value range T1 — min=0.0000  max=0.4991
[T5.1] Value range T2 — min=0.0000  max=0.4976


In [6]:
## 5.2 — NDVI Computation & Differencing

# %%
def compute_ndvi(stack: np.ndarray, nir_idx: int = 6, red_idx: int = 2) -> np.ndarray:
    """Compute per-pixel NDVI from a (H,W,C) reflectance stack.

    NDVI = (NIR − Red) / (NIR + Red + ε)
    """
    nir = stack[..., nir_idx].astype(np.float64)
    red = stack[..., red_idx].astype(np.float64)
    ndvi = (nir - red) / (nir + red + 1e-10)
    return np.clip(ndvi, -1.0, 1.0).astype(np.float32)


def ndvi_change_detection(
    ndvi_t1: np.ndarray,
    ndvi_t2: np.ndarray,
    threshold: float = 0.15,
    adaptive: bool = True,
) -> tuple[np.ndarray, np.ndarray]:
    """Detect NDVI drops between two time periods.

    Args:
        ndvi_t1 / ndvi_t2 : (H,W) arrays
        threshold          : fixed threshold; overridden when adaptive=True
        adaptive           : use Otsu-like percentile threshold if True

    Returns:
        change_mask : bool (H,W) — True where vegetation is stressed
        change_mag  : float (H,W) — magnitude of NDVI decline
    """
    change_mag = ndvi_t1 - ndvi_t2      # positive = vegetation loss

    if adaptive:
        positive_vals = change_mag[change_mag > 0]
        if positive_vals.size > 0:
            threshold = float(np.percentile(positive_vals, 70))
            print(f"[T5.2] Adaptive threshold (70th-pct of positive Δ): {threshold:.4f}")
        else:
            print(f"[T5.2] No positive change found; using fixed threshold={threshold}")

    change_mask = change_mag > threshold
    print(f"[T5.2] Stressed pixels: {change_mask.sum()} / {change_mask.size} "
          f"({100*change_mask.mean():.1f}%)")
    return change_mask, change_mag


# ── Compute NDVI ──────────────────────────────────────────────────────────────
ndvi_t1 = compute_ndvi(stack_t1)
ndvi_t2 = compute_ndvi(stack_t2)

print(f"[T5.2] NDVI T1 — mean={ndvi_t1.mean():.4f}  std={ndvi_t1.std():.4f}")
print(f"[T5.2] NDVI T2 — mean={ndvi_t2.mean():.4f}  std={ndvi_t2.std():.4f}")

ndvi_change_mask, ndvi_change_mag = ndvi_change_detection(
    ndvi_t1, ndvi_t2, threshold=0.15, adaptive=True
)

[T5.2] NDVI T1 — mean=0.3149  std=0.5749
[T5.2] NDVI T2 — mean=0.2804  std=0.5567
[T5.2] Adaptive threshold (70th-pct of positive Δ): 0.2333
[T5.2] Stressed pixels: 10746 / 65536 (16.4%)


In [7]:
 ## 5.3 — Log-Ratio Change Maps (Multi-Band)

# %%
def log_ratio_change(stack_t1: np.ndarray, stack_t2: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    """Compute per-band log-ratio change map.

    LR(b) = log(stack_t2[b] / stack_t1[b])  per pixel
    Large |LR| values indicate spectral change.

    Returns:
        lr_map : (H, W, C) float32 — log-ratio per band
    """
    t1_safe = np.clip(stack_t1, eps, None)
    t2_safe = np.clip(stack_t2, eps, None)
    lr_map  = np.log(t2_safe / t1_safe).astype(np.float32)
    return lr_map


def log_ratio_magnitude(lr_map: np.ndarray) -> np.ndarray:
    """L2-norm of log-ratio across all bands → scalar change magnitude (H,W)."""
    return np.linalg.norm(lr_map, axis=2).astype(np.float32)


lr_map      = log_ratio_change(stack_t1, stack_t2)
lr_magnitude = log_ratio_magnitude(lr_map)

# Threshold at 99th percentile for binary log-ratio mask
lr_thresh     = float(np.percentile(lr_magnitude, 99))
lr_change_mask = lr_magnitude > lr_thresh

print(f"[T5.3] Log-ratio magnitude — mean={lr_magnitude.mean():.4f}  max={lr_magnitude.max():.4f}")
print(f"[T5.3] Log-ratio threshold (99th-pct): {lr_thresh:.4f}")
print(f"[T5.3] Changed pixels: {lr_change_mask.sum()} ({100*lr_change_mask.mean():.2f}%)")

[T5.3] Log-ratio magnitude — mean=3.4040  max=24.7421
[T5.3] Log-ratio threshold (99th-pct): 19.0657
[T5.3] Changed pixels: 656 (1.00%)


In [8]:
# ## 5.4 — Change Vector Analysis (CVA)
#
# CVA computes the **Euclidean distance** (magnitude) and **direction** of change
# in N-dimensional spectral feature space between T1 and T2.

# %%
def change_vector_analysis(
    stack_t1: np.ndarray,
    stack_t2: np.ndarray,
    nir_idx: int = 6,
    red_idx: int = 2,
) -> tuple[np.ndarray, np.ndarray]:
    """Compute CVA magnitude and direction across all spectral bands.

    Args:
        stack_t1/t2 : (H, W, C) float32 multi-band stacks
        nir_idx     : band index for NIR (for 2D direction projection)
        red_idx     : band index for Red

    Returns:
        magnitude : (H,W) — Euclidean distance in C-dim feature space
        direction : (H,W) — arctan2 angle (radians) in NIR–Red subspace
    """
    diff      = stack_t2.astype(np.float32) - stack_t1.astype(np.float32)   # (H,W,C)
    magnitude = np.linalg.norm(diff, axis=2)                                  # (H,W)
    direction = np.arctan2(diff[..., nir_idx], diff[..., red_idx])            # (H,W)
    return magnitude.astype(np.float32), direction.astype(np.float32)


def cva_change_mask(
    magnitude: np.ndarray,
    pct_threshold: float = 95.0,
) -> np.ndarray:
    """Threshold CVA magnitude at a given percentile to produce binary mask."""
    thresh = float(np.percentile(magnitude, pct_threshold))
    mask   = magnitude > thresh
    print(f"[T5.4] CVA magnitude threshold ({pct_threshold}th-pct): {thresh:.4f}")
    print(f"[T5.4] CVA changed pixels: {mask.sum()} ({100*mask.mean():.2f}%)")
    return mask


cva_magnitude, cva_direction = change_vector_analysis(stack_t1, stack_t2)
cva_mask = cva_change_mask(cva_magnitude, pct_threshold=95.0)

print(f"[T5.4] CVA magnitude — mean={cva_magnitude.mean():.4f}  max={cva_magnitude.max():.4f}")

[T5.4] CVA magnitude threshold (95.0th-pct): 0.2252
[T5.4] CVA changed pixels: 3277 (5.00%)
[T5.4] CVA magnitude — mean=0.0604  max=0.2656


In [9]:
 ## 5.5 — Ensemble Change Mask (Fusion of NDVI + LR + CVA)

# %%
def ensemble_change_mask(
    ndvi_mask: np.ndarray,
    lr_mask:   np.ndarray,
    cva_mask:  np.ndarray,
    min_votes: int = 2,
) -> np.ndarray:
    """Majority-vote fusion of three change detectors.

    A pixel is flagged as 'changed' if at least `min_votes` detectors agree.
    This reduces both false alarms (from single detectors) and missed detections.
    """
    vote_map = ndvi_mask.astype(np.int8) + lr_mask.astype(np.int8) + cva_mask.astype(np.int8)
    fused    = vote_map >= min_votes
    print(f"[T5.5] Ensemble mask (≥{min_votes}/3 detectors agree): "
          f"{fused.sum()} pixels ({100*fused.mean():.2f}%)")
    return fused


# ── Morphological clean-up ────────────────────────────────────────────────────
def morpho_clean(mask: np.ndarray, min_area_px: int = 25) -> np.ndarray:
    """Remove small speckle polygons (< min_area_px pixels) via connected-component filtering."""
    labeled, n_comp = scipy_label(mask)
    cleaned = np.zeros_like(mask, dtype=bool)
    for i in range(1, n_comp + 1):
        component = labeled == i
        if component.sum() >= min_area_px:
            cleaned |= component
    print(f"[T5.5] Morpho clean — components before: {n_comp}  |  kept: {cleaned.sum()} px")
    return cleaned


fused_mask = ensemble_change_mask(ndvi_change_mask, lr_change_mask, cva_mask, min_votes=2)
final_mask = morpho_clean(fused_mask, min_area_px=25)


[T5.5] Ensemble mask (≥2/3 detectors agree): 3133 pixels (4.78%)
[T5.5] Morpho clean — components before: 858  |  kept: 926 px


In [10]:
## 5.6 — Siamese CNN (PyTorch) for Deep Change Detection
#
# Architecture: Twin ResNet-style encoder sharing weights → difference feature maps
# → sigmoid output (change probability per pixel).
# Trained on synthetic patch pairs; in production use LEVIR-CD or S2Looking.

# %%
# ── Siamese encoder block ─────────────────────────────────────────────────────
class ConvBnRelu(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, kernel: int = 3, padding: int = 1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel, padding=padding, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)


class SiameseEncoder(nn.Module):
    """Lightweight shared-weight encoder for a single temporal branch."""
    def __init__(self, in_channels: int = 10):
        super().__init__()
        self.enc1 = ConvBnRelu(in_channels, 32)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = ConvBnRelu(32, 64)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = ConvBnRelu(64, 128)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        return e1, e2, e3   # multi-scale features


class SiameseCNN(nn.Module):
    """Full Siamese Change Detection Network.

    Input : x_t1, x_t2 — (B, C, H, W) tensor pairs
    Output: change_prob — (B, 1, H, W) sigmoid probability map
    """
    def __init__(self, in_channels: int = 10):
        super().__init__()
        self.encoder = SiameseEncoder(in_channels)

        # Decoder with skip connections from difference features
        self.up2   = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.dec2  = ConvBnRelu(128 + 64, 64)

        self.up1   = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.dec1  = ConvBnRelu(64  + 32, 32)

        self.head  = nn.Conv2d(32, 1, kernel_size=1)   # logit map

    def forward(self, x_t1: torch.Tensor, x_t2: torch.Tensor) -> torch.Tensor:
        # Shared-weight encoding
        e1_t1, e2_t1, e3_t1 = self.encoder(x_t1)
        e1_t2, e2_t2, e3_t2 = self.encoder(x_t2)

        # Absolute difference features at each scale
        d1 = torch.abs(e1_t1 - e1_t2)   # (B, 32, H,   W)
        d2 = torch.abs(e2_t1 - e2_t2)   # (B, 64, H/2, W/2)
        d3 = torch.abs(e3_t1 - e3_t2)   # (B, 128,H/4, W/4)

        # Decoder: upsample + skip
        x = self.up2(d3)
        x = self.dec2(torch.cat([x, d2], dim=1))

        x = self.up1(x)
        x = self.dec1(torch.cat([x, d1], dim=1))

        return torch.sigmoid(self.head(x))   # (B, 1, H, W)


# ── Synthetic patch dataset ───────────────────────────────────────────────────
class SyntheticChangePatchDataset(Dataset):
    """Generate synthetic (T1, T2, label) patch pairs on-the-fly for training demo."""
    def __init__(self, n_samples: int = 200, patch_size: int = 64, channels: int = 10):
        self.n       = n_samples
        self.ps      = patch_size
        self.C       = channels
        self.rng     = np.random.default_rng(7)

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        ps, C = self.ps, self.C
        rng = np.random.default_rng(idx)

        t1 = rng.random((C, ps, ps), dtype=np.float32)
        t2 = t1.copy()
        label = np.zeros((1, ps, ps), dtype=np.float32)

        # 50% of patches contain a change region
        if rng.random() > 0.5:
            r0, c0 = rng.integers(0, ps//2, size=2)
            rh, rw = rng.integers(10, ps//2, size=2)
            t2[:, r0:r0+rh, c0:c0+rw] += rng.random((C, rh, rw), dtype=np.float32) * 0.6
            label[0, r0:r0+rh, c0:c0+rw] = 1.0

        t1 = np.clip(t1, 0, 1)
        t2 = np.clip(t2, 0, 1)

        return (
            torch.from_numpy(t1),
            torch.from_numpy(t2),
            torch.from_numpy(label),
        )


# ── Training loop ─────────────────────────────────────────────────────────────
def train_siamese_cnn(
    model:       SiameseCNN,
    n_epochs:    int = 5,
    batch_size:  int = 8,
    lr:          float = 1e-3,
    n_samples:   int = 200,
    patch_size:  int = 64,
) -> list[float]:
    """Train the Siamese CNN on synthetic patches and return per-epoch loss list."""
    ds     = SyntheticChangePatchDataset(n_samples, patch_size)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True, num_workers=0)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCELoss()
    model.to(DEVICE).train()

    loss_history = []
    print(f"\n[T5.6] Training Siamese CNN on {DEVICE} — {n_epochs} epochs × {len(ds)} samples")
    print("-" * 60)

    for epoch in range(1, n_epochs + 1):
        epoch_loss = 0.0
        for t1_batch, t2_batch, lbl_batch in loader:
            t1_batch  = t1_batch.to(DEVICE)
            t2_batch  = t2_batch.to(DEVICE)
            lbl_batch = lbl_batch.to(DEVICE)

            optimizer.zero_grad()
            pred = model(t1_batch, t2_batch)
            loss = criterion(pred, lbl_batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        avg = epoch_loss / len(loader)
        loss_history.append(avg)
        print(f"  Epoch {epoch:02d}/{n_epochs}  BCE Loss: {avg:.6f}")

    print("-" * 60)
    return loss_history


siamese_model = SiameseCNN(in_channels=C)
loss_history  = train_siamese_cnn(siamese_model, n_epochs=5, batch_size=8)


# ── Inference on full scene ───────────────────────────────────────────────────
def siamese_inference(
    model:   SiameseCNN,
    t1:      np.ndarray,
    t2:      np.ndarray,
    threshold: float = 0.50,
) -> tuple[np.ndarray, np.ndarray]:
    """Run trained Siamese CNN on full-scene arrays (H,W,C) → change prob + binary mask."""
    model.eval().to(DEVICE)
    with torch.no_grad():
        t1_t = torch.from_numpy(t1.transpose(2, 0, 1)).unsqueeze(0).to(DEVICE)   # (1,C,H,W)
        t2_t = torch.from_numpy(t2.transpose(2, 0, 1)).unsqueeze(0).to(DEVICE)
        prob = model(t1_t, t2_t).squeeze().cpu().numpy()                           # (H,W)
    cnn_mask = prob > threshold
    print(f"[T5.6] Siamese CNN — changed px: {cnn_mask.sum()} ({100*cnn_mask.mean():.2f}%)")
    return prob, cnn_mask


cnn_prob, cnn_mask = siamese_inference(siamese_model, stack_t1, stack_t2, threshold=0.50)



[T5.6] Training Siamese CNN on cuda — 5 epochs × 200 samples
------------------------------------------------------------
  Epoch 01/5  BCE Loss: 0.447287
  Epoch 02/5  BCE Loss: 0.338551
  Epoch 03/5  BCE Loss: 0.253894
  Epoch 04/5  BCE Loss: 0.192196
  Epoch 05/5  BCE Loss: 0.148919
------------------------------------------------------------
[T5.6] Siamese CNN — changed px: 0 (0.00%)


In [11]:
## 5.7 — Synthetic Ground-Truth Reference Mask
#
# In production: load from a manually annotated GeoTIFF or shapefile.

# %%
# Reference: the injected stress region is the ground-truth "changed" area
ref_mask = np.zeros((H, W), dtype=bool)
ref_mask[STRESS_RS, STRESS_CS] = True
print(f"[T5.7] Reference changed pixels: {ref_mask.sum()} ({100*ref_mask.mean():.2f}%)")

[T5.7] Reference changed pixels: 6400 (9.77%)


In [12]:
## 5.8 — Evaluation: Producer's Acc · User's Acc · OA · Cohen's κ · FAR · MDR

# %%
def evaluate_change_detection(
    pred_mask: np.ndarray,
    ref_mask:  np.ndarray,
    name:      str = "model",
) -> dict:
    """Compute full change-detection evaluation metrics.

    Returns dict with keys: kappa, oa, pa, ua, far, mdr
    """
    pred = pred_mask.ravel().astype(int)
    ref  = ref_mask.ravel().astype(int)

    # Handle edge case: both arrays are identical class
    if pred.sum() == 0 and ref.sum() == 0:
        print(f"[T5.8] {name}: trivial all-zero prediction — κ undefined (set to 0)")
        return dict(kappa=0, oa=1, pa=0, ua=0, far=0, mdr=0)

    kappa = cohen_kappa_score(ref, pred)
    cm    = confusion_matrix(ref, pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    total = cm.sum()
    oa    = (tp + tn) / total                      if total          > 0 else 0.0
    pa    = tp / (tp + fn)                         if (tp + fn)      > 0 else 0.0   # recall
    ua    = tp / (tp + fp)                         if (tp + fp)      > 0 else 0.0   # precision
    far   = fp / (fp + tn)                         if (fp + tn)      > 0 else 0.0
    mdr   = fn / (fn + tp)                         if (fn + tp)      > 0 else 0.0

    print(
        f"[T5.8] {name:20s} | κ={kappa:.4f}  OA={oa:.4f}  "
        f"PA={pa:.4f}  UA={ua:.4f}  FAR={far:.4f}  MDR={mdr:.4f}"
    )
    return dict(kappa=kappa, oa=oa, pa=pa, ua=ua, far=far, mdr=mdr)


print("\n[T5.8] === Change Detection Evaluation ===")
metrics_ndvi     = evaluate_change_detection(ndvi_change_mask, ref_mask, "NDVI Differencing")
metrics_lr       = evaluate_change_detection(lr_change_mask,   ref_mask, "Log-Ratio")
metrics_cva      = evaluate_change_detection(cva_mask,         ref_mask, "CVA")
metrics_ensemble = evaluate_change_detection(final_mask,       ref_mask, "Ensemble (fused)")
metrics_cnn      = evaluate_change_detection(cnn_mask,         ref_mask, "Siamese CNN")

all_metrics = {
    "NDVI Differencing" : metrics_ndvi,
    "Log-Ratio"         : metrics_lr,
    "CVA"               : metrics_cva,
    "Ensemble (fused)"  : metrics_ensemble,
    "Siamese CNN"       : metrics_cnn,
}


[T5.8] === Change Detection Evaluation ===
[T5.8] NDVI Differencing    | κ=0.5720  OA=0.9017  PA=0.8364  UA=0.4981  FAR=0.0912  MDR=0.1636
[T5.8] Log-Ratio            | κ=-0.0185  OA=0.8923  PA=0.0000  UA=0.0000  FAR=0.0111  MDR=1.0000
[T5.8] CVA                  | κ=0.6544  OA=0.9523  PA=0.5120  UA=1.0000  FAR=0.0000  MDR=0.4880
[T5.8] Ensemble (fused)     | κ=0.2339  OA=0.9165  PA=0.1447  UA=1.0000  FAR=0.0000  MDR=0.8553
[T5.8] Siamese CNN          | κ=0.0000  OA=0.9023  PA=0.0000  UA=0.0000  FAR=0.0000  MDR=1.0000


In [13]:
## 5.9 — Output A: Export GeoTIFF Binary Change Mask (EPSG:32642)

# %%
def export_change_geotiff(
    mask:       np.ndarray,
    transform,
    crs_str:    str,
    out_path:   Path,
    nodata:     int = 255,
) -> None:
    """Write a binary change mask (bool/int) to a single-band GeoTIFF.

    Values: 0 = no change, 1 = stressed / changed
    """
    mask_u8 = mask.astype(np.uint8)
    profile = {
        "driver"   : "GTiff",
        "dtype"    : "uint8",
        "width"    : mask.shape[1],
        "height"   : mask.shape[0],
        "count"    : 1,
        "crs"      : CRS.from_string(crs_str),
        "transform": transform,
        "compress" : "lzw",
        "nodata"   : nodata,
    }
    with rasterio.open(out_path, "w", **profile) as dst:
        dst.write(mask_u8[np.newaxis, ...])

    size_kb = out_path.stat().st_size / 1024
    print(f"[T5.9] GeoTIFF saved → {out_path}  ({size_kb:.1f} KB)")


geotiff_path = OUT / "task5_change_mask.tif"
export_change_geotiff(final_mask, TRANSFORM, EPSG_UTM, geotiff_path)



[T5.9] GeoTIFF saved → task5_outputs/task5_change_mask.tif  (2.2 KB)


In [14]:
# ## 5.10 — Output B: Shapefile of Alert Polygons (GeoPandas/Fiona)

# %%
def classify_severity(mean_mag: float) -> str:
    """Severity classification based on mean NDVI change magnitude in a polygon."""
    if mean_mag > 0.30:
        return "Severe"
    elif mean_mag > 0.20:
        return "Moderate"
    else:
        return "Mild"


def generate_alert_shapefile(
    change_mask: np.ndarray,
    transform,
    crs_str:     str,
    change_mag:  np.ndarray,
    output_path: Path,
    min_area_ha: float = 0.1,
) -> gpd.GeoDataFrame | None:
    """Vectorise binary change mask and export alert polygons as Shapefile.

    Polygon attributes:
        area_ha       : area in hectares (computed in EPSG:32642)
        severity      : 'Mild' | 'Moderate' | 'Severe'
        mean_mag      : mean NDVI drop within polygon
        centroid_lat  : WGS-84 latitude of centroid
        centroid_lon  : WGS-84 longitude of centroid
        alert_id      : sequential polygon identifier
    """
    mask_u8 = change_mask.astype(np.uint8)
    crs_obj = CRS.from_string(crs_str)

    # Extract polygons from rasterised mask (rasterio shapes)
    polys = [
        {"geometry": shape(geom), "value": val}
        for geom, val in shapes(mask_u8, mask=mask_u8, transform=transform)
        if val == 1
    ]

    if not polys:
        print("[T5.10] No stressed areas detected above threshold.")
        return None

    gdf = gpd.GeoDataFrame(polys, crs=crs_obj)
    gdf.set_geometry("geometry", inplace=True)

    # Reproject to UTM for metric area computation
    gdf_utm = gdf.to_crs(EPSG_UTM)
    gdf_utm["area_ha"] = gdf_utm.geometry.area / 10_000.0

    # Filter tiny fragments
    gdf_utm = gdf_utm[gdf_utm["area_ha"] >= min_area_ha].copy()
    if gdf_utm.empty:
        print(f"[T5.10] All polygons < {min_area_ha} ha — no alerts.")
        return None

    # Per-polygon mean NDVI change magnitude (sample from raster)
    gdf_utm["mean_mag"]  = np.nan
    gdf_utm["alert_id"]  = range(1, len(gdf_utm) + 1)

    for idx in gdf_utm.index:
        geom = gdf_utm.at[idx, "geometry"]
        # Approximate centroid pixel lookup in change_mag array
        from rasterio.transform import rowcol
        cx, cy = geom.centroid.x, geom.centroid.y
        try:
            row, col = rowcol(transform, cx, cy)
            row = int(np.clip(row, 0, change_mag.shape[0] - 1))
            col = int(np.clip(col, 0, change_mag.shape[1] - 1))
            gdf_utm.at[idx, "mean_mag"] = float(change_mag[row, col])
        except Exception:
            gdf_utm.at[idx, "mean_mag"] = 0.0

    gdf_utm["severity"] = gdf_utm["mean_mag"].apply(classify_severity)

    # GPS centroid (WGS-84) for field navigation
    gdf_wgs = gdf_utm.to_crs("EPSG:4326")
    gdf_utm["centroid_lat"] = gdf_wgs.geometry.centroid.y
    gdf_utm["centroid_lon"] = gdf_wgs.geometry.centroid.x

    # Shapefile export (drop raw 'value' column — not needed)
    gdf_export = gdf_utm.drop(columns=["value"], errors="ignore")
    gdf_export.to_file(output_path, driver="ESRI Shapefile")

    print(f"[T5.10] Alert shapefile saved → {output_path}")
    print(f"[T5.10] Polygons exported: {len(gdf_export)}")
    print(f"[T5.10] Severity summary:\n{gdf_utm['severity'].value_counts().to_string()}")
    print(f"\n[T5.10] Alert table:")
    print(
        gdf_utm[["alert_id", "area_ha", "severity", "mean_mag",
                 "centroid_lat", "centroid_lon"]].to_string(index=False)
    )
    return gdf_utm


shp_path = OUT / "task5_alert_polygons.shp"
gdf_alerts = generate_alert_shapefile(
    final_mask,
    TRANSFORM,
    EPSG_UTM,
    ndvi_change_mag,
    shp_path,
    min_area_ha=0.01,
)


[T5.10] All polygons < 0.01 ha — no alerts.


In [15]:
 ## 5.11 — Output C: Matplotlib Stakeholder Map Overlay

# %%
def make_stakeholder_map(
    stack_t1:      np.ndarray,
    stack_t2:      np.ndarray,
    ndvi_t1:       np.ndarray,
    ndvi_t2:       np.ndarray,
    change_mag:    np.ndarray,
    final_mask:    np.ndarray,
    cnn_prob:      np.ndarray,
    gdf_alerts:    gpd.GeoDataFrame | None,
    loss_history:  list[float],
    metrics_dict:  dict,
    out_path:      Path,
) -> None:
    """6-panel stakeholder report figure.

    Panels:
      [0,0] False-colour composite T1 (NIR-Red-Green)
      [0,1] False-colour composite T2
      [1,0] NDVI difference map (Δ NDVI = T1-T2)
      [1,1] CVA magnitude map
      [2,0] Final ensemble change mask with alert polygon overlays
      [2,1] Siamese CNN change probability + training loss inset
    """
    fig = plt.figure(figsize=(18, 20), facecolor="#0F1117")
    gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.38, wspace=0.22)

    AX_TITLE_KW  = dict(color="white", fontsize=13, fontweight="bold", pad=10)
    CB_LABEL_KW  = dict(color="#CCCCCC", fontsize=9)
    BORDER_ALPHA = 0.3

    SEVERITY_COLORS = {"Mild": "#FFD700", "Moderate": "#FF8C00", "Severe": "#FF2222"}

    # ── RGB helper ────────────────────────────────────────────────────────────
    def to_rgb(stack, r_idx=6, g_idx=2, b_idx=1):
        """Build a display RGB from band indices; auto-stretch per channel."""
        def stretch(arr):
            lo, hi = np.percentile(arr, 2), np.percentile(arr, 98)
            return np.clip((arr - lo) / (hi - lo + 1e-10), 0, 1)
        return np.stack([stretch(stack[..., i]) for i in (r_idx, g_idx, b_idx)], axis=-1)

    rgb_t1 = to_rgb(stack_t1)
    rgb_t2 = to_rgb(stack_t2)

    # ── Panel 0,0: False-colour T1 ────────────────────────────────────────────
    ax00 = fig.add_subplot(gs[0, 0])
    ax00.imshow(rgb_t1, origin="upper")
    ax00.set_title("T1 — False Colour (NIR-Red-Green)", **AX_TITLE_KW)
    ax00.axis("off")
    ax00.text(0.02, 0.97, "Pre-stress", transform=ax00.transAxes,
              color="#AAFFAA", fontsize=9, va="top", ha="left")

    # ── Panel 0,1: False-colour T2 ────────────────────────────────────────────
    ax01 = fig.add_subplot(gs[0, 1])
    ax01.imshow(rgb_t2, origin="upper")
    ax01.set_title("T2 — False Colour (NIR-Red-Green)", **AX_TITLE_KW)
    ax01.axis("off")
    ax01.text(0.02, 0.97, "Post-stress", transform=ax01.transAxes,
              color="#FFAAAA", fontsize=9, va="top", ha="left")

    # Annotate stress region on T2
    import matplotlib.patches as patches
    r0, c0 = STRESS_RS.start, STRESS_CS.start
    rh = STRESS_RS.stop - STRESS_RS.start
    rw = STRESS_CS.stop - STRESS_CS.start
    rect = patches.Rectangle((c0, r0), rw, rh, linewidth=2,
                               edgecolor="#FF4444", facecolor="none", linestyle="--")
    ax01.add_patch(rect)
    ax01.text(c0 + rw // 2, r0 - 5, "Stress Zone", color="#FF4444",
              fontsize=8, ha="center")

    # ── Panel 1,0: NDVI difference map ───────────────────────────────────────
    ax10 = fig.add_subplot(gs[1, 0])
    vmax = max(abs(change_mag.min()), abs(change_mag.max()), 0.01)
    im10 = ax10.imshow(change_mag, cmap="RdYlGn_r", vmin=-vmax, vmax=vmax, origin="upper")
    ax10.set_title("ΔNDVI (T1 − T2)  [Positive = Stress]", **AX_TITLE_KW)
    ax10.axis("off")
    cb10 = plt.colorbar(im10, ax=ax10, fraction=0.046, pad=0.04)
    cb10.set_label("NDVI Drop Magnitude", **CB_LABEL_KW)
    cb10.ax.yaxis.set_tick_params(color="#CCCCCC")
    plt.setp(cb10.ax.yaxis.get_ticklabels(), color="#CCCCCC")

    # ── Panel 1,1: CVA magnitude map ─────────────────────────────────────────
    ax11 = fig.add_subplot(gs[1, 1])
    im11 = ax11.imshow(cva_magnitude, cmap="plasma", origin="upper")
    ax11.set_title("CVA Magnitude (all bands)", **AX_TITLE_KW)
    ax11.axis("off")
    cb11 = plt.colorbar(im11, ax=ax11, fraction=0.046, pad=0.04)
    cb11.set_label("Euclidean Distance (spectral space)", **CB_LABEL_KW)
    cb11.ax.yaxis.set_tick_params(color="#CCCCCC")
    plt.setp(cb11.ax.yaxis.get_ticklabels(), color="#CCCCCC")

    # Overlay CVA mask contour
    ax11.contour(cva_mask.astype(float), levels=[0.5], colors=["#00FFFF"], linewidths=1)

    # ── Panel 2,0: Ensemble mask + alert polygons ─────────────────────────────
    ax20 = fig.add_subplot(gs[2, 0])
    ax20.imshow(rgb_t2, origin="upper", alpha=0.55)

    # Overlay binary change mask
    cmap_mask = ListedColormap(["none", "#FF220033"])
    ax20.imshow(final_mask, cmap=cmap_mask, origin="upper", alpha=0.7)

    # Plot alert polygon outlines
    legend_patches = []
    if gdf_alerts is not None and not gdf_alerts.empty:
        for sev, col in SEVERITY_COLORS.items():
            sub = gdf_alerts[gdf_alerts["severity"] == sev]
            if sub.empty:
                continue
            for geom in sub.geometry:
                if geom.geom_type == "Polygon":
                    xs, ys = geom.exterior.coords.xy
                    # Convert world coords to pixel coords (approximate)
                    col_coords = [(c - TRANSFORM.c) / TRANSFORM.a for c in xs]
                    row_coords = [(r - TRANSFORM.f) / TRANSFORM.e for r in ys]
                    ax20.plot(col_coords, row_coords, color=col, linewidth=1.5)
            legend_patches.append(mpatches.Patch(color=col, label=f"{sev} stress"))

    ax20.set_title("Ensemble Change Mask + Alert Polygons", **AX_TITLE_KW)
    ax20.axis("off")
    if legend_patches:
        ax20.legend(handles=legend_patches, loc="lower right",
                    facecolor="#222222", labelcolor="white", fontsize=8)

    # Metric text
    m = metrics_dict.get("Ensemble (fused)", {})
    metric_str = (
        f"κ={m.get('kappa',0):.3f}  OA={m.get('oa',0):.3f}\n"
        f"PA={m.get('pa',0):.3f}  UA={m.get('ua',0):.3f}\n"
        f"FAR={m.get('far',0):.3f}  MDR={m.get('mdr',0):.3f}"
    )
    ax20.text(0.02, 0.03, metric_str, transform=ax20.transAxes,
              color="white", fontsize=8, va="bottom",
              bbox=dict(facecolor="#111111", alpha=0.7, boxstyle="round,pad=0.3"))

    # ── Panel 2,1: Siamese CNN prob + training loss inset ────────────────────
    ax21 = fig.add_subplot(gs[2, 1])
    im21 = ax21.imshow(cnn_prob, cmap="hot", vmin=0, vmax=1, origin="upper")
    ax21.set_title("Siamese CNN — Change Probability", **AX_TITLE_KW)
    ax21.axis("off")
    cb21 = plt.colorbar(im21, ax=ax21, fraction=0.046, pad=0.04)
    cb21.set_label("P(Change)", **CB_LABEL_KW)
    cb21.ax.yaxis.set_tick_params(color="#CCCCCC")
    plt.setp(cb21.ax.yaxis.get_ticklabels(), color="#CCCCCC")

    # Inset: training loss curve
    ax_ins = ax21.inset_axes([0.58, 0.58, 0.40, 0.38])
    ax_ins.plot(range(1, len(loss_history) + 1), loss_history,
                color="#00EEFF", linewidth=1.5, marker="o", markersize=3)
    ax_ins.set_facecolor("#111111")
    ax_ins.tick_params(colors="white", labelsize=6)
    ax_ins.set_title("Train Loss", color="white", fontsize=7, pad=2)
    ax_ins.spines[:].set_color("#444444")
    for label in ax_ins.get_xticklabels() + ax_ins.get_yticklabels():
        label.set_color("white")

    # Metric overlay
    m_cnn = metrics_dict.get("Siamese CNN", {})
    cnn_metric_str = (
        f"κ={m_cnn.get('kappa',0):.3f}  OA={m_cnn.get('oa',0):.3f}\n"
        f"PA={m_cnn.get('pa',0):.3f}  UA={m_cnn.get('ua',0):.3f}"
    )
    ax21.text(0.02, 0.03, cnn_metric_str, transform=ax21.transAxes,
              color="white", fontsize=8, va="bottom",
              bbox=dict(facecolor="#111111", alpha=0.7, boxstyle="round,pad=0.3"))

    # ── Figure title ──────────────────────────────────────────────────────────
    fig.suptitle(
        "AgriSense — Task 5: Multi-Temporal Change Detection & Crop Stress Alerts\n"
        f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}  |  "
        f"AOI: Punjab, Pakistan  |  EPSG:32642",
        color="white", fontsize=15, fontweight="bold", y=0.99,
    )

    fig.savefig(out_path, dpi=150, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    plt.close(fig)
    size_kb = out_path.stat().st_size / 1024
    print(f"[T5.11] Stakeholder map saved → {out_path}  ({size_kb:.1f} KB)")


map_path = OUT / "task5_stakeholder_map.png"
make_stakeholder_map(
    stack_t1, stack_t2, ndvi_t1, ndvi_t2,
    ndvi_change_mag, final_mask, cnn_prob,
    gdf_alerts, loss_history, all_metrics, map_path,
)

[T5.11] Stakeholder map saved → task5_outputs/task5_stakeholder_map.png  (3345.8 KB)


In [16]:
## 5.12 — CSV Metrics Report

# %%
def save_metrics_csv(metrics_dict: dict, out_path: Path) -> None:
    """Write all per-method evaluation metrics to a CSV file."""
    fieldnames = ["method", "kappa", "oa", "pa", "ua", "far", "mdr"]
    with open(out_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for method, m in metrics_dict.items():
            writer.writerow({
                "method" : method,
                "kappa"  : f"{m['kappa']:.4f}",
                "oa"     : f"{m['oa']:.4f}",
                "pa"     : f"{m['pa']:.4f}",
                "ua"     : f"{m['ua']:.4f}",
                "far"    : f"{m['far']:.4f}",
                "mdr"    : f"{m['mdr']:.4f}",
            })
    print(f"[T5.12] Metrics CSV saved → {out_path}")


csv_path = OUT / "task5_metrics.csv"
save_metrics_csv(all_metrics, csv_path)

[T5.12] Metrics CSV saved → task5_outputs/task5_metrics.csv


In [17]:
## 5.13 — Alert Threshold Sensitivity Analysis (NDVI drop > 0.15 / 10-day window)

# %%
def alert_threshold_sensitivity(
    ndvi_t1:   np.ndarray,
    ndvi_t2:   np.ndarray,
    ref_mask:  np.ndarray,
    thresholds: list[float] | None = None,
) -> None:
    """Sweep NDVI drop thresholds and report κ, PA, UA, FAR at each level.

    The guide specifies threshold = 0.15 within a 10-day revisit window.
    This function validates that choice against the full performance curve.
    """
    if thresholds is None:
        thresholds = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]

    print("\n[T5.13] NDVI Threshold Sensitivity (10-day window spec: 0.15)")
    print(f"{'Threshold':>10}  {'κ':>7}  {'OA':>7}  {'PA':>7}  {'UA':>7}  {'FAR':>7}  {'MDR':>7}")
    print("-" * 65)

    rows = []
    for thr in thresholds:
        mask, _ = ndvi_change_detection(ndvi_t1, ndvi_t2, threshold=thr, adaptive=False)
        m = evaluate_change_detection(mask, ref_mask, name=f"thr={thr:.2f}")
        flag = " ← spec" if abs(thr - 0.15) < 1e-6 else ""
        print(f"{thr:>10.2f}  {m['kappa']:>7.4f}  {m['oa']:>7.4f}  "
              f"{m['pa']:>7.4f}  {m['ua']:>7.4f}  {m['far']:>7.4f}  {m['mdr']:>7.4f}{flag}")
        rows.append({"threshold": thr, **m})

    # Save sensitivity CSV
    sens_path = OUT / "task5_threshold_sensitivity.csv"
    with open(sens_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["threshold", "kappa", "oa", "pa", "ua", "far", "mdr"])
        writer.writeheader()
        for row in rows:
            writer.writerow({k: (f"{v:.4f}" if isinstance(v, float) else v) for k, v in row.items()})
    print(f"[T5.13] Sensitivity CSV saved → {sens_path}")


alert_threshold_sensitivity(ndvi_t1, ndvi_t2, ref_mask)





[T5.13] NDVI Threshold Sensitivity (10-day window spec: 0.15)
 Threshold        κ       OA       PA       UA      FAR      MDR
-----------------------------------------------------------------
[T5.2] Stressed pixels: 21060 / 65536 (32.1%)
[T5.8] thr=0.05             | κ=0.3676  OA=0.7747  PA=0.9919  UA=0.3014  FAR=0.2488  MDR=0.0081
      0.05   0.3676   0.7747   0.9919   0.3014   0.2488   0.0081
[T5.2] Stressed pixels: 14382 / 65536 (21.9%)
[T5.8] thr=0.10             | κ=0.5410  OA=0.8741  PA=0.9791  UA=0.4357  FAR=0.1372  MDR=0.0209
      0.10   0.5410   0.8741   0.9791   0.4357   0.1372   0.0209
[T5.2] Stressed pixels: 12536 / 65536 (19.1%)
[T5.8] thr=0.15             | κ=0.5900  OA=0.8969  PA=0.9513  UA=0.4856  FAR=0.1090  MDR=0.0488
      0.15   0.5900   0.8969   0.9513   0.4856   0.1090   0.0488 ← spec
[T5.2] Stressed pixels: 11526 / 65536 (17.6%)
[T5.8] thr=0.20             | κ=0.5876  OA=0.9014  PA=0.8955  UA=0.4972  FAR=0.0980  MDR=0.1045
      0.20   0.5876   0.9014   0.895

In [18]:
 ## 5.14 — Summary Console Report

# %%
def print_summary(metrics_dict: dict, gdf_alerts: gpd.GeoDataFrame | None) -> None:
    """Print the comprehensive Task 5 summary table to console."""
    SEP = "=" * 90

    print(f"\n{SEP}")
    print("  AgriSense — Task 5 Comprehensive Metrics Report")
    print(f"  Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
    print(SEP)

    header = f"{'Method':<22} {'κ':>7} {'OA':>7} {'PA':>7} {'UA':>7} {'FAR':>7} {'MDR':>7}"
    print(header)
    print("-" * len(header))
    for method, m in metrics_dict.items():
        print(
            f"{method:<22} {m['kappa']:>7.4f} {m['oa']:>7.4f} {m['pa']:>7.4f} "
            f"{m['ua']:>7.4f} {m['far']:>7.4f} {m['mdr']:>7.4f}"
        )

    print(f"\n{SEP}")
    print("  Alert Polygon Summary")
    print(SEP)
    if gdf_alerts is not None and not gdf_alerts.empty:
        print(f"  Total polygons  : {len(gdf_alerts)}")
        print(f"  Total area (ha) : {gdf_alerts['area_ha'].sum():.2f}")
        print(f"  Severity counts : {gdf_alerts['severity'].value_counts().to_dict()}")
        print(f"  Largest polygon : {gdf_alerts['area_ha'].max():.4f} ha")
    else:
        print("  No alert polygons generated.")

    print(f"\n{SEP}")
    print("  Output Files")
    print(SEP)
    for f in sorted(OUT.iterdir()):
        print(f"  {f.name:<45} {f.stat().st_size / 1024:>8.1f} KB")
    print(SEP + "\n")


print_summary(all_metrics, gdf_alerts)


  AgriSense — Task 5 Comprehensive Metrics Report
  Generated: 2026-06-05 15:45
Method                       κ      OA      PA      UA     FAR     MDR
----------------------------------------------------------------------
NDVI Differencing       0.5720  0.9017  0.8364  0.4981  0.0912  0.1636
Log-Ratio              -0.0185  0.8923  0.0000  0.0000  0.0111  1.0000
CVA                     0.6544  0.9523  0.5120  1.0000  0.0000  0.4880
Ensemble (fused)        0.2339  0.9165  0.1447  1.0000  0.0000  0.8553
Siamese CNN             0.0000  0.9023  0.0000  0.0000  0.0000  1.0000

  Alert Polygon Summary
  No alert polygons generated.

  Output Files
  task5_change_mask.tif                              2.2 KB
  task5_metrics.csv                                  0.3 KB
  task5_stakeholder_map.png                       3345.8 KB
  task5_threshold_sensitivity.csv                    0.3 KB

